# Two-joint arm

Two boards, two joints, `coaxial.motion.servo` on each.

In [1]:
SIMULATED = True          # False, and PORT, at the bench
PORT = 'COM4'

One board per joint: unit 1 the shoulder, unit 2 the elbow. On a bus they share the segment; on the stand-in each unit is its own board.

In [2]:
from coaxial import Coaxial63100

shoulder = Coaxial63100(port=PORT, unit=1, simulated_device=SIMULATED).open()
elbow = Coaxial63100(port=PORT, unit=2, simulated_device=SIMULATED).open()
for joint in (shoulder, elbow):
    joint.drive.source('model')
    joint.drive.model_param(j=2e-5, b=1e-5, load=0.0)
    joint.gates.arm(bypass_sto=True, ignore_interlock=True)
print(shoulder, elbow)

<Coaxial63100 Simulated SIMULATED> <Coaxial63100 Simulated SIMULATED>


A move is a pair of targets; each servo slews, settles, reads its shaft and corrects what the load stole. The elbow carries a standing load, which is what a link hanging off it is.

In [3]:
elbow.drive.model_param(load=0.01)
POSES = ((30.0, 60.0), (60.0, 20.0), (0.0, 0.0))
reached = []
with shoulder.motion.servo(amps=2.0) as s, elbow.motion.servo(amps=2.0) as e:
    for a, b in POSES:
        got_a = s.to(a, tol=0.5)
        got_b = e.to(b, tol=0.5)
        reached.append((a, b, got_a, s.error, got_b, e.error))
        print('pose (%5.1f, %5.1f)  shoulder %6.2f err %5.2f  elbow %6.2f err %5.2f'
              % (a, b, got_a, s.error, got_b, e.error))
for joint in (shoulder, elbow):
    joint.gates.disarm()
    joint.drive.model_param(load=0.0)
    joint.drive.source('adc')
    joint.close()

pose ( 30.0,  60.0)  shoulder  29.98 err  0.02  elbow  59.90 err  0.10


pose ( 60.0,  20.0)  shoulder  60.06 err -0.06  elbow  20.05 err -0.05


pose (  0.0,   0.0)  shoulder   0.10 err -0.10  elbow   0.31 err -0.31


## Conclusions

In [4]:
print('%-16s %-22s %-22s' % ('pose', 'shoulder', 'elbow (0.01 N.m)'))
for a, b, got_a, err_a, got_b, err_b in reached:
    print('(%5.1f, %5.1f)   %7.2f deg err %5.2f   %7.2f deg err %5.2f'
          % (a, b, got_a, err_a, got_b, err_b))
print()
print('worst shoulder error %.2f deg, worst elbow error %.2f deg'
      % (max(abs(r[3]) for r in reached), max(abs(r[5]) for r in reached)))
print('tolerance asked for  0.50 deg, up to 4 corrections a move')

pose             shoulder               elbow (0.01 N.m)      
( 30.0,  60.0)     29.98 deg err  0.02     59.90 deg err  0.10
( 60.0,  20.0)     60.06 deg err -0.06     20.05 deg err -0.05
(  0.0,   0.0)      0.10 deg err -0.10      0.31 deg err -0.31

worst shoulder error 0.10 deg, worst elbow error 0.31 deg
tolerance asked for  0.50 deg, up to 4 corrections a move


Two boards, two unit ids, one segment. A Modbus RTU frame carries the unit id first and every node on the wire sees every frame; `bus_message` counts what passed and `server_message` only what was addressed here, so the difference is the traffic meant for the other joint. Unit 0 is broadcast - every node acts, none answers, and reads are refused.

Each joint holds its own current vector, so nothing about the pair is coupled through the drive: what couples them is the arm, and the elbow's standing load is what its own servo corrects out. The correction is what the load stole - a spring wound by holding torque, or poles slipped outright.